# Interpretation of Results

## Comprehensive Summary of Results

### Were we successful in modeling life expectancy?

Our project set out to answer whether PM2.5 air pollution is a significant predictor of life expectancy across countries, and the results suggest a cautiously affirmative answer - though the story is more nuanced than a simple bivariate relationship.

We built two OLS models. **Model 1 (baseline)** regressed life expectancy on log-transformed PM2.5 alone, yielding an R² of 0.183. While statistically significant (p < 0.001), this model explains less than one-fifth of the variation in life expectancy, confirming that PM2.5 alone is insufficient for prediction.

**Model 2 (controlled model)** added log GDP per capita and log health expenditure per capita as covariates, raising R² dramatically to 0.704. On the held-out 20% test set, we achieved:

| Metric | Training Set | Test Set |
|--------|-------------|----------|
| R²     | 0.708       | 0.687    |
| RMSE   | 4.85 years  | 4.99 years |

The small gap between training and test performance (~0.02 in R², ~0.14 years in RMSE) suggests limited overfitting. However, since a random row-level split was used, observations from the same country appear in both train and test sets, which may inflate test performance. Results should therefore be interpreted with caution regarding generalization to truly unseen countries.

### Real-World Interpretation of Parameters (Model 2)

The final model is:

$$\hat{LifeExp} = 26.742 - 0.370 \cdot \log(PM_{2.5}) + 2.828 \cdot \log(GDP) + 2.971 \cdot \log(HealthExp)$$

Each coefficient has a meaningful real-world interpretation:

- **log_PM25 (β = −0.370, p = 0.014):** Holding GDP and health expenditure constant, a 1% increase in PM2.5 concentration is associated with a decrease of approximately 0.004 years (≈ 1.4 days) in life expectancy. A *doubling* of PM2.5 corresponds to roughly 0.26 fewer years of life expectancy (β × ln(2) = −0.370 × 0.693 ≈ −0.257 years), or about 3 months. This aligns with the epidemiological literature linking chronic fine particulate exposure to cardiovascular and respiratory disease mortality.

- **log_GDP (β = 2.828, p < 0.001):** A doubling of GDP per capita is associated with approximately 1.96 additional years of life expectancy, net of pollution and health spending. This captures the broad infrastructure, food security, and institutional quality benefits of economic development.

- **log_HealthExp (β = 2.971, p < 0.001):** A doubling of health expenditure per capita is associated with approximately 2.06 additional years of life expectancy. This is the strongest individual predictor, reflecting how healthcare access, quality, and coverage translate directly into longevity outcomes.

### Key Insights

1. **Omitted variable bias in Model 1:** The baseline coefficient for log_PM25 was -6.255, nearly 17x larger than in the controlled model (-0.370). This dramatic reduction reveals that raw PM2.5 was acting as a proxy for poverty - poorer countries tend to have both worse air quality and lower life expectancy for unrelated reasons. Once economic controls are added, the isolated effect of PM2.5 becomes much more modest, though still statistically significant.

2. **Confounding between GDP and health expenditure:** The correlation between log_GDP and log_HealthExp was r = 0.95, indicating near-multicollinearity. Wealthier nations naturally spend more on healthcare, making it difficult to disentangle their individual contributions. The coefficients should be interpreted jointly rather than in strict isolation.

3. **Persistent PM2.5-life expectancy gap:** The EDA showed a persistent gap in life expectancy between high-PM2.5 and low-PM2.5 country groups across the entire 2000-2020 period, ranging from approximately 5.5 to 7.7 years with an average of about 7 years, even as global PM2.5 levels declined overall. This suggests structural and systemic disparities that persist beyond year-to-year pollution fluctuations.

## Strengths of the Model

**1. Low overfitting on the training split**

The model achieved R² = 0.687 on the held-out test set, close to the training R² = 0.708. This small gap suggests the model is not heavily overfit to the training data. Note that because a random row-level split was used rather than a country-level split, this should not be taken as strong evidence of generalization to entirely new countries. An RMSE of approximately 5 years on a target variable spanning roughly 70 years is a respectable performance for a three-predictor OLS model.

**2. Interpretability and theoretical grounding**

By using OLS with log-transformed predictors, all coefficients carry clear semi-elasticity interpretations. The direction and relative magnitude of each predictor (GDP, health expenditure, PM2.5) are consistent with established economic and public health theory - wealthier, healthier-spending countries live longer, and cleaner air confers an independent longevity benefit. This interpretability is a major advantage over "black-box" models and is particularly suited to the policy-relevant context of this study.

**3. Heteroscedasticity-robust standard errors**

The use of HC1 robust standard errors helps ensure that our p-values and confidence intervals remain valid in the presence of heteroscedasticity across countries, which box plots and scatter plots suggested was likely. Note that HC1 addresses heteroscedasticity only and does not correct for the serial correlation present in panel data; this remains a limitation addressed in Section 8.3.

## Shortcomings of the Model

**1. Temporal autocorrelation and panel structure ignored**

The Durbin-Watson statistic for both models was approximately 0.11-0.15, far below the acceptable threshold of ~2.0. This indicates severe positive serial autocorrelation - successive years for the same country produce residuals that are highly correlated. The model treats 3,416 country-year observations as independent, but in reality, a country's life expectancy in 2015 is heavily determined by its life expectancy in 2014. This violates a core OLS assumption, inflates effective sample size, and likely overstates the precision of our estimates.

**2. Residual non-normality**

The Jarque-Bera test was highly significant in both models (p ≈ 0 in Model 2), indicating that the residuals are not normally distributed - they are left-skewed (skew ≈ -1.56) with heavy tails (kurtosis ≈ 7.6). This is partly attributable to a small number of extreme outliers (e.g., countries with life expectancy below 45 years due to conflict or epidemic, such as Lesotho and Central African Republic). While OLS coefficient estimates remain unbiased, hypothesis tests and confidence intervals are less reliable under severe non-normality in smaller subsamples.

**3. Omitted variables and reverse causality**

Despite controlling for GDP and health expenditure, important confounders remain unaccounted for: political stability, urbanization rates, educational attainment, disease burden (e.g., HIV/AIDS prevalence in Sub-Saharan Africa), and access to clean water. Additionally, reverse causality may exist - longer-lived, healthier populations may be more productive, generating higher GDP. A true causal claim about PM2.5's effect on life expectancy would require an instrumental variable or natural experiment design, which is beyond the scope of this OLS analysis.

## Recommendations for Future Researchers

**What would we do differently, and what advice do we offer?**

1. **Use a panel fixed-effects model.** Incorporating country fixed effects would control for all time-invariant country characteristics (geography, culture, institutional quality) that are omitted from our model, reducing omitted variable bias. Note that fixed effects alone do not fully resolve serial autocorrelation; pairing this approach with HAC (Newey-West) standard errors or a dynamic panel estimator would be needed to properly address temporal dependence.

2. **Include additional health and demographic controls.** Future studies should consider adding variables such as HIV prevalence, urbanization rate, years of schooling, and access to safe water. These variables explain variation in life expectancy that is currently absorbed into residuals, reducing RMSE and improving coefficient reliability.

3. **Expand PM2.5 data granularity.** Our analysis used country-level annual averages, which masks significant within-country variation - urban populations may face PM2.5 concentrations 3-5x higher than rural populations in the same country. Sub-national or city-level data would allow for a more precise estimate of pollution's health impact.

4. **Consider non-linear modeling for outliers.** The residual non-normality suggests that life expectancy responds non-linearly to predictors at extreme values. A quantile regression approach or generalized additive model (GAM) could better capture the dynamics for the lowest-income, highest-pollution countries that currently produce the most extreme residuals.

5. **Address the direction of causality.** To make credible causal claims, future work should seek instrumental variables for PM2.5 - for example, wind patterns or geographic features that affect pollution transport but not life expectancy directly. This would allow for stronger policy recommendations beyond correlation.